<!-- NB_DOC_INTRO_V1 -->
# NLP — Classification supervisee (TF-IDF, BoW, sklearn)

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Classification de texte** = predire la classe d'un document (spam / non-spam, sentiment, sujet, ...). Approche classique : **vectorisation** (BoW, TF-IDF, n-grams) + **classifieur sklearn** (LogReg, SVM, NaiveBayes, RF). Toujours une solide **baseline** avant de sortir BERT (souvent suffit a 80-95% de la qualite, 100x plus rapide).

Ce notebook execute le bench complet sur **20 Newsgroups** (auto-DL via sklearn).

> ⚠️ Typo dans le nom du fichier (`Spervisé` au lieu de `Supervisé`) — conservee pour compatibilite.

## Plan

1. Setup + dataset 20 Newsgroups
2. Cleaning (tokenization, stopwords, lemmatisation)
3. BoW (CountVectorizer)
4. TF-IDF (TfidfVectorizer)
5. Bench : NaiveBayes, LogReg, SVM, RF
6. Pipeline + GridSearchCV
7. N-grams (impact)
8. Modeles top-tier 2024-2025 (alternatives)
9. Pieges + References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
import time
import warnings
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 1. Dataset — 20 Newsgroups (subset 4 categories pour rapidite)

In [ ]:
categories = ["sci.med", "sci.space", "rec.sport.hockey", "comp.graphics"]
data_tr = fetch_20newsgroups(subset="train", categories=categories,
                              remove=("headers", "footers", "quotes"), random_state=SEED)
data_te = fetch_20newsgroups(subset="test", categories=categories,
                              remove=("headers", "footers", "quotes"), random_state=SEED)

print(f"Train : {len(data_tr.data)} docs, Classes : {data_tr.target_names}")
print(f"Test  : {len(data_te.data)} docs")
print(f"\n--- Exemple ---\n{data_tr.data[0][:400]}...")
print(f"\nLabel : {data_tr.target_names[data_tr.target[0]]}")

## 2. BoW (Bag-of-Words)

In [ ]:
bow = CountVectorizer(stop_words="english", max_features=10_000, min_df=3)
X_tr_bow = bow.fit_transform(data_tr.data)
X_te_bow = bow.transform(data_te.data)
print(f"BoW shape : {X_tr_bow.shape}  (sparse {X_tr_bow.nnz / np.prod(X_tr_bow.shape):.2%})")
print(f"Vocab : {len(bow.vocabulary_)} tokens")
print(f"Sample tokens : {list(bow.vocabulary_)[:10]}")

## 3. TF-IDF (recommande, baseline solide)

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=10_000, min_df=3,
                          ngram_range=(1, 2), sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(data_tr.data)
X_te_tfidf = tfidf.transform(data_te.data)
print(f"TF-IDF shape : {X_tr_tfidf.shape}")

## 4. Bench 4 modeles

In [ ]:
y_tr = data_tr.target
y_te = data_te.target

models = {
    "MultinomialNB":   MultinomialNB(),
    "LogReg":          LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
    "LinearSVC":       LinearSVC(C=1.0, random_state=SEED),
    "RandomForest":    RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
}

results = []
for name, clf in models.items():
    t0 = time.perf_counter()
    clf.fit(X_tr_tfidf, y_tr)
    fit_t = time.perf_counter() - t0
    pred = clf.predict(X_te_tfidf)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average="macro")
    results.append({"model": name, "test_acc": acc, "test_f1_macro": f1, "fit_time_s": fit_t})

df_r = pd.DataFrame(results).sort_values("test_f1_macro", ascending=False)
print(df_r.round(4))

## 5. Pipeline + GridSearchCV

In [ ]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf",   LogisticRegression(max_iter=1000, random_state=SEED)),
])

param_grid = {
    "tfidf__max_features": [5_000, 10_000],
    "tfidf__ngram_range":  [(1, 1), (1, 2)],
    "tfidf__min_df":       [2, 5],
    "clf__C":              [0.1, 1.0, 10.0],
}
gs = GridSearchCV(pipe, param_grid, cv=3, scoring="f1_macro", n_jobs=-1, verbose=0)
gs.fit(data_tr.data, y_tr)
print(f"Best params : {gs.best_params_}")
print(f"Best CV F1  : {gs.best_score_:.4f}")
pred = gs.predict(data_te.data)
print(f"Test F1 macro : {f1_score(y_te, pred, average='macro'):.4f}")
print(classification_report(y_te, pred, target_names=data_tr.target_names, digits=3))

## 6. Top tokens predictifs par classe (LogReg interpretable)

In [ ]:
best_pipe = gs.best_estimator_
clf = best_pipe.named_steps["clf"]
vocab = np.array(best_pipe.named_steps["tfidf"].get_feature_names_out())

# Pour multi-classe, clf.coef_ shape = (n_classes, n_features)
for idx, cls in enumerate(data_tr.target_names):
    top = np.argsort(clf.coef_[idx])[-10:][::-1]
    print(f"\n{cls} : {', '.join(vocab[top])}")

## 7. Modeles top-tier 2024-2025 (alternatives a TF-IDF)

| Approche | Forces | Limites |
|---|---|---|
| **TF-IDF + LinearSVC / LogReg** (ici) | Baseline rapide, interpretable, ~ 80-90% acc | Pas de semantique (synonymes) |
| **Sentence-Transformers + LogReg** | Embeddings semantiques, peu de tuning | 80MB modele + 1ere passe lente |
| **SetFit** (HuggingFace) | Few-shot, marche bien sur 10-100 exemples | Plus complexe |
| **DistilBERT fine-tuned** | Top qualite | Gros (250MB), lent, GPU recommande |
| **Cohere / OpenAI embedding API** | Zero train, ultra simple | Cout, latence reseau |

Voir [NLP_Recherche_d_informations.ipynb](./NLP_Recherche_d_informations.ipynb) pour les embeddings.

## 8. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Reporter accuracy sur 4 categories mais 1 class 80% | F1 macro |
| TF-IDF sans `min_df` | Vocab geant, overfit |
| Pas `stop_words` | Bruit |
| `RandomForest` sur sparse TF-IDF | Tres lent et sub-optimal → LinearSVC / LogReg |
| Stemming agressif (Porter) | Perd info — preferer lemmatisation ou rien |
| Pas tester n-grams | Bigrams souvent +2-5% F1 |
| GridSearch avec 10 params | RandomizedSearch ou Optuna |
| 1 langue, 1 modele | Si multi-langue, utiliser modele multi-langue |


## References

### Documentation
- sklearn text : https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- NLTK : https://www.nltk.org/
- spaCy : https://spacy.io/

### Voir aussi
- [NLP_Classification_Smote.ipynb](NLP_Classification_Smote.ipynb)
- [NLP_Recherche_d_informations.ipynb](NLP_Recherche_d_informations.ipynb)
- [NLP_Transformers.ipynb](NLP_Transformers.ipynb)
- [AI_Prompt_Engineering.ipynb](AI_Prompt_Engineering.ipynb)
